<a href="https://colab.research.google.com/github/Zenith23B/Portfolio-Risk-Engine-Bryan-Alexandre/blob/main/Portfolio_Risk_Engine_Bryan_Alexandre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Moteur d'Évaluation du Risque de Portefeuille
### Analyse quantitative d'actifs individuels et de fonds indiciels vectorisés (ETF)

Ce document présente un outil d'analyse et de gestion des risques financiers, structuré de manière autonome pour évaluer un portefeuille d'actifs financiers diversifiés (incluant des actions américaines d'envergure, des obligations souveraines et des indices sectoriels clés comme le *iShares Semiconductor ETF* - SOXX).

#### Fonctionnalités principales :
* **Extraction quantitative :** Récupération automatisée des cours historiques de clôture ajustés via l'API Yahoo Finance.
* **Statistiques descriptives :** Calcul des rendements arithmétiques, de la volatilité annualisée, du ratio de Sharpe empirique et des drawdowns historiques.
* **Mesures de risques de marché :** Estimation de la Value-at-Risk (VaR) et de l'Expected Shortfall (ES) selon trois approches méthodologiques complémentaires : historique, paramétrique (loi normale multivariée) et par simulations de Monte Carlo.
* **Contrôle et validation :** Backtesting statistique de la validité de la VaR quotidienne via le test de couverture inconditionnelle de Kupiec.
* **Stress Testing & Analyse de Scénarios :** Simulation de chocs historiques réels (Crise financière de 2008, krach de la Covid-19 en 2020) et de scénarios de stress sur la matrice de corrélation.

*Note méthodologique :* Les calculs sont effectués en dollar américain (USD) sur la base de cours ajustés des dividendes et divisions d'actions (`auto_adjust=True`). Le modèle suppose un rééquilibrage quotidien aux poids cibles sans frottements ni coûts de transaction.

## 1. Installation des bibliothèques scientifiques

In [1]:
!pip -q install "yfinance>=0.2.54,<2" "numpy>=1.26,<3" "pandas>=2.1,<3" "scipy>=1.11,<2" "plotly>=5.20,<7"

## 2. Définition des fonctions et du moteur quantitatif

Cette section regroupe les fonctions requises pour le traitement des données et les calculs de risque :

* **Value-at-Risk (VaR) :** Représente le seuil de perte maximale toléré sur un horizon temporel donné, à un niveau de confiance fixé (noté $\alpha$).
* **Expected Shortfall (ES) :** Évalue la perte moyenne attendue au-delà du seuil déterminé par la VaR.
* **Test de Kupiec :** Test statistique de ratio de vraisemblance vérifiant si le taux d'exceptions observé durant la période de backtesting dévie de manière significative du taux attendu ($1 - \alpha$).

In [2]:
from __future__ import annotations
import io, json, math, zipfile
from dataclasses import dataclass
from datetime import date, timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
import yfinance as yf
from scipy.stats import chi2, norm
TRADING_DAYS = 252
UNIVERSE = {
    "Actions US": {
        "AAPL": "Apple", "MSFT": "Microsoft", "NVDA": "NVIDIA",
        "AMZN": "Amazon", "GOOGL": "Alphabet", "JPM": "JPMorgan Chase",
        "JNJ": "Johnson & Johnson", "XOM": "Exxon Mobil",
    },
    "ETF actions / secteurs": {
        "SPY": "S&P 500", "QQQ": "Nasdaq 100", "SOXX": "iShares Semiconductor ETF",
        "IWM": "Russell 2000", "EFA": "Actions pays developpes hors US",
        "EEM": "Actions emergentes",
    },
    "ETF obligataires": {
        "TLT": "Treasuries US 20+ ans", "IEF": "Treasuries US 7-10 ans",
        "BND": "Obligations US aggregate",
    },
    "Autres classes (ETF)": {
        "GLD": "Or", "VNQ": "Immobilier cote US", "DBC": "Panier de futures matieres premieres",
    },
}
ALL_TICKERS = [ticker for items in UNIVERSE.values() for ticker in items]
NAMES = {ticker: name for items in UNIVERSE.values() for ticker, name in items.items()}
CATEGORIES = {ticker: group for group, items in UNIVERSE.items() for ticker in items}
DEFAULT_WEIGHTS = {
    "SPY": .20, "QQQ": .10, "SOXX": .10, "AAPL": .10, "NVDA": .10,
    "TLT": .15, "BND": .10, "GLD": .10, "EFA": .05,
}
CRISES = {
    "Crise financiere 2008": ("2008-09-01", "2009-03-09"),
    "Covid 2020": ("2020-02-19", "2020-03-23"),
    "Marches 2022": ("2022-01-03", "2022-10-12"),
}


@dataclass(frozen=True)
class Risk:
    var: float  # fraction of portfolio value; reported as positive loss
    es: float   # fraction of portfolio value; reported as positive loss


def fmt_pct(value: float) -> str:
    return f"{value:.2%}" if np.isfinite(value) else "N/D"


def fmt_usd(value: float) -> str:
    return f"${value:,.0f}" if np.isfinite(value) else "N/D"


def download_prices(tickers: tuple[str, ...], start: str, end_exclusive: str) -> pd.DataFrame:
    """Adjusted close via auto_adjust; date de fin yfinance exclusive."""
    raw = yf.download(
        tickers=list(tickers), start=start, end=end_exclusive,
        auto_adjust=True, progress=False, threads=True,
        group_by="column", actions=False,
    )
    if raw is None or raw.empty:
        raise ValueError("Yahoo Finance n'a retourne aucune donnee pour cette selection.")
    if isinstance(raw.columns, pd.MultiIndex):
        # yfinance avec group_by='column' -> premier niveau Price
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"].copy()
        elif "Close" in raw.columns.get_level_values(1):
            close = raw.xs("Close", axis=1, level=1).copy()
        else:
            raise ValueError("La colonne Close ajustee est introuvable.")
    else:
        if "Close" not in raw.columns:
            raise ValueError("La colonne Close ajustee est introuvable.")
        close = raw[["Close"]].rename(columns={"Close": tickers[0]})
    if isinstance(close, pd.Series):
        close = close.to_frame(name=tickers[0])
    close.columns = close.columns.astype(str).str.upper()
    missing = sorted(set(tickers) - set(close.columns))
    if missing:
        raise ValueError(f"Actifs non telecharges : {', '.join(missing)}")
    close = close.loc[:, list(tickers)].sort_index()
    close.index = pd.to_datetime(close.index).tz_localize(None).normalize()
    close = close.apply(pd.to_numeric, errors="coerce").replace(0, np.nan)
    return close


def clean_prices(prices: pd.DataFrame, strict: bool = True) -> tuple[pd.DataFrame, pd.Series]:
    """Aucune interpolation de prix: intersection des dates avec cours valides."""
    before = len(prices)
    counts = prices.notna().sum()
    for col in prices:
        if counts[col] < 252:
            raise ValueError(f"Historique insuffisant pour {col} ({counts[col]} observations).")
    if strict:
        result = prices.dropna(how="any")
    else:
        # FFill borne a 2 seances, jamais de backward fill.
        result = prices.ffill(limit=2).dropna(how="any")
    if len(result) < 252:
        raise ValueError("Moins de 252 dates alignees. Reduisez l'univers ou changez la periode.")
    removed = before - len(result)
    return result, pd.Series({"initial": before, "retenues": len(result), "supprimees": removed})


def portfolio_returns(returns: pd.DataFrame, weights: np.ndarray) -> pd.Series:
    """Portefeuille a poids cibles constants: r_p,t = somme(w_i*r_i,t)."""
    return pd.Series(returns.to_numpy() @ weights, index=returns.index, name="Portfolio")


def risk_from_returns(scenario_returns: np.ndarray, confidence: float) -> Risk:
    """VaR et ES empiriques, avec interpolation d'une fraction de l'observation de seuil.

    Quantile des pertes a alpha. ES = moyenne de la queue de masse 1-alpha,
    avec contribution fractionnaire de l'observation limite (ES >= VaR si queue negative).
    Valeurs negatives possibles si *tous* les scenarios gagnent: ne pas masquer
    cette information avec un floor artificiel a zero.
    """
    x = np.asarray(scenario_returns, dtype=float).reshape(-1)
    x = x[np.isfinite(x)]
    if len(x) < 20:
        raise ValueError("Trop peu de scenarios pour estimer VaR/ES.")
    losses = np.sort(-x)[::-1]  # pertes de la plus forte a la plus faible
    tail_n = (1.0 - confidence) * len(losses)
    var = float(np.quantile(losses, confidence))
    nwhole = int(np.floor(tail_n))
    frac = tail_n - nwhole
    tail_sum = float(losses[:nwhole].sum())
    if frac > 1e-12:
        tail_sum += frac * float(losses[nwhole])
    es = tail_sum / tail_n
    # Par construction il faut que ES >= VaR, sauf epsilon numerique.
    return Risk(var=var, es=max(float(es), var))


def parametric_risk(mu_daily: float, sigma_daily: float, confidence: float, horizon: int) -> Risk:
    """Approximation normale des rendements simples cumules sur h jours.

    mu_h = h*mu_1, sigma_h = sqrt(h)*sigma_1 ; pas de correction compounding.
    """
    mu_h = mu_daily * horizon
    sigma_h = sigma_daily * np.sqrt(horizon)
    z = norm.ppf(confidence)
    var = z * sigma_h - mu_h
    es = sigma_h * norm.pdf(z) / (1.0 - confidence) - mu_h
    return Risk(float(var), float(es))


def historical_horizon_returns(returns: pd.Series, horizon: int) -> np.ndarray:
    """Rendements composes glissants (fenetres chevauchantes si horizon>1)."""
    return ((1.0 + returns).rolling(horizon).apply(np.prod, raw=True) - 1.0).dropna().to_numpy()


def mc_multivariate(
    returns: pd.DataFrame, weights: np.ndarray, horizon: int,
    n_sim: int, rng: np.random.Generator,
) -> np.ndarray:
    """Simulations gaussiennes multivariees des rendements simples des actifs.

    Covariance reguliere PSD via projection spectrale; simulations par lots
    pour limiter la RAM. Portefeuille reequilibre a ses poids cibles chaque jour.
    """
    mu = returns.mean().to_numpy()
    covariance = returns.cov().to_numpy()
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    factor = eigenvectors @ np.diag(np.sqrt(np.maximum(eigenvalues, 0.0)))
    out = np.empty(n_sim, dtype=float)
    batch = 2000
    for begin in range(0, n_sim, batch):
        size = min(batch, n_sim - begin)
        normals = rng.standard_normal((size, horizon, len(weights)))
        asset_scenarios = normals @ factor.T + mu
        daily_port = asset_scenarios @ weights
        out[begin:begin + size] = np.prod(1.0 + daily_port, axis=1) - 1.0
    return out


def drawdown(port: pd.Series) -> pd.Series:
    cumulative = (1.0 + port).cumprod()
    return cumulative / cumulative.cummax() - 1.0


def portfolio_statistics(ret: pd.DataFrame, weights: np.ndarray, rf: float) -> tuple[pd.DataFrame, dict]:
    port = portfolio_returns(ret, weights)
    annual_return = float((1.0 + port).prod() ** (TRADING_DAYS / len(port)) - 1.0)
    log_return = float(np.log1p(port).mean() * TRADING_DAYS)
    sigma = float(port.std(ddof=1) * np.sqrt(TRADING_DAYS))
    sharpe = (float(port.mean()) * TRADING_DAYS - rf) / sigma if sigma > 0 else np.nan
    cov = ret.cov().to_numpy() * TRADING_DAYS
    p_vol = float(np.sqrt(max(weights @ cov @ weights, 0.0)))
    marginal = (cov @ weights) / p_vol if p_vol > 0 else np.zeros_like(weights)
    contrib = weights * marginal
    metrics = pd.DataFrame({
        "Actif": ret.columns,
        "Poids": weights,
        "Rendement annuel (geometrique)": ((1 + ret).prod() ** (TRADING_DAYS / len(ret)) - 1).to_numpy(),
        "Volatilite annuelle": (ret.std(ddof=1) * np.sqrt(TRADING_DAYS)).to_numpy(),
        "Contribution vol (points/an)": contrib,
        "Part du risque": contrib / p_vol if p_vol > 0 else np.nan,
    }).set_index("Actif")
    summary = {
        "Rendement annualise geometrique": annual_return,
        "Rendement log annualise": log_return,
        "Volatilite annualisee": sigma,
        "Sharpe annualise": sharpe,
        "Drawdown maximum": float(drawdown(port).min()),
        "Variance via covariance": float(weights @ cov @ weights),
    }
    return metrics, summary


def kupiec_test(exceptions: int, observations: int, confidence: float) -> tuple[float, float]:
    """Kupiec POF LRuc, chi-deux 1 ddl; clipping pour x=0 ou x=n."""
    p0 = 1.0 - confidence
    x, n = exceptions, observations
    if n == 0:
        return np.nan, np.nan
    phat = x / n
    def xlogy(xcount: int, p: float) -> float:
        if xcount == 0:
            return 0.0
        return xcount * math.log(p)
    log_null = xlogy(x, p0) + xlogy(n - x, 1.0 - p0)
    log_alt = xlogy(x, phat) + xlogy(n - x, 1.0 - phat)
    statistic = max(0.0, -2.0 * (log_null - log_alt))
    return statistic, float(chi2.sf(statistic, df=1))


def backtest_var(returns: pd.DataFrame, weights: np.ndarray, confidence: float,
                 rolling_window: int, method: str = "Historique") -> pd.DataFrame:
    """VaR estimee sur t-W,...,t-1 et testee contre la perte realisee a t."""
    port = portfolio_returns(returns, weights)
    data = []
    for t in range(rolling_window, len(port)):
        history = port.iloc[t - rolling_window:t].to_numpy()
        if method == "Historique":
            var = risk_from_returns(history, confidence).var
        else:
            var = parametric_risk(float(history.mean()), float(history.std(ddof=1)), confidence, 1).var
        actual_loss = -float(port.iloc[t])
        data.append((port.index[t], actual_loss, var, actual_loss > var))
    return pd.DataFrame(data, columns=["Date", "Perte realisee", "VaR prevue", "Exception"]).set_index("Date")


def traffic_light(backtest: pd.DataFrame, confidence: float) -> str:
    """Seuils Basel standards uniquement pour 250 observations / VaR 99 %."""
    if abs(confidence - .99) > 1e-9 or len(backtest) != 250:
        return "Non applicable (feu tricolore standard: 250 jours, VaR 99 %)."
    x = int(backtest["Exception"].sum())
    return "Vert (0-4)" if x <= 4 else ("Jaune (5-9)" if x <= 9 else "Rouge (10+)")


def historical_crisis(prices: pd.DataFrame, weights: np.ndarray, start: str, end: str,
                      required_coverage: float = .85) -> dict | None:
    """Replay buy-and-hold: exige que les cours couvrent l'essentiel de la fenetre."""
    panel = prices.loc[start:end]
    if len(panel) < 5:
        return None
    start_target = pd.Timestamp(start)
    end_target = pd.Timestamp(end)
    if panel.index.min() > start_target + pd.Timedelta(days=12):
        return None
    if panel.index.max() < end_target - pd.Timedelta(days=12):
        return None
    all_business_days = len(pd.bdate_range(start_target, end_target))
    if len(panel) / max(all_business_days, 1) < required_coverage:
        return None
    asset_shocks = (panel.iloc[-1] / panel.iloc[0] - 1.0).to_numpy()
    contributions = asset_shocks * weights
    return {
        "return": float(contributions.sum()),
        "worst": str(prices.columns[np.argmin(contributions)]),
        "contributions": pd.Series(contributions, index=prices.columns),
        "from": panel.index[0], "to": panel.index[-1],
    }


def stressed_correlation_cov(covariance: np.ndarray, rho: float) -> np.ndarray:
    """Correlation equicorrelee: PSD pour 0<=rho<=1."""
    sigmas = np.sqrt(np.clip(np.diag(covariance), 0, None))
    corr = np.full_like(covariance, rho, dtype=float)
    np.fill_diagonal(corr, 1.0)
    return np.outer(sigmas, sigmas) * corr


def pack_outputs(prices: pd.DataFrame, returns: pd.DataFrame, risk_table: pd.DataFrame,
                 positions: pd.DataFrame, backtest: pd.DataFrame, metadata: dict) -> bytes:
    output = io.BytesIO()
    with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name, frame in {
            "prix_ajustes_usd.csv": prices, "rendements_journaliers.csv": returns,
            "risque_var_es.csv": risk_table, "positions_et_contributions.csv": positions,
            "backtest.csv": backtest,
        }.items():
            archive.writestr(name, frame.to_csv(index=True).encode("utf-8-sig"))
        archive.writestr("parametres.json", json.dumps(metadata, indent=2, ensure_ascii=False, default=str))
    return output.getvalue()




## 3. Configuration des paramètres de l'étude

Définition de l'allocation d'actifs, de la période d'échantillonnage historique, du niveau de capital initial ainsi que des paramètres de confiance et d'horizons temporels pour l'évaluation des risques.

In [3]:
# ---------- PARAMÈTRES UTILISATEUR ----------
WEIGHTS = {
    "SPY": 0.20, "QQQ": 0.10, "SOXX": 0.10,
    "AAPL": 0.10, "NVDA": 0.10,
    "TLT": 0.15, "BND": 0.10, "GLD": 0.10, "EFA": 0.05,
}
START_DATE = "2019-01-01"
END_DATE_EXCLUSIVE = (date.today() + timedelta(days=1)).isoformat()
INITIAL_CAPITAL_USD = 1_000_000
RISK_FREE_RATE_ANNUAL = 0.03   # Hypothèse pédagogique fixe, pas un taux live
CONFIDENCE_LEVELS = (0.95, 0.975, 0.99)
HORIZON_DAYS = 1               # 1, 5 ou 10 par exemple
N_MONTE_CARLO = 30_000         # Ajuster selon la RAM Colab
RANDOM_SEED = 42
BACKTEST_WINDOW = 500
BACKTEST_CONFIDENCE = 0.99
STRICT_DATES = True            # Supprime toutes les dates incomplètes

assert len(WEIGHTS) >= 6, "Sélectionner au moins 6 actifs."
assert all(t in ALL_TICKERS for t in WEIGHTS), "Ticker non reconnu dans l'univers."
assert all(w >= 0 for w in WEIGHTS.values()), "Les poids doivent être positifs."
assert np.isclose(sum(WEIGHTS.values()), 1.0), "La somme des poids doit être égale à 1."
assert 0.0 < BACKTEST_CONFIDENCE < 1.0
assert HORIZON_DAYS >= 1 and N_MONTE_CARLO >= 1000
TICKERS = list(WEIGHTS)
W = np.array([WEIGHTS[t] for t in TICKERS], dtype=float)
print('Actifs:', TICKERS)
print('SOXX :', f'{WEIGHTS.get("SOXX", 0):.0%}')
print('Classes présentes :', sorted({CATEGORIES[t] for t in TICKERS}))


Actifs: ['SPY', 'QQQ', 'SOXX', 'AAPL', 'NVDA', 'TLT', 'BND', 'GLD', 'EFA']
SOXX : 10%
Classes présentes : ['Actions US', 'Autres classes (ETF)', 'ETF actions / secteurs', 'ETF obligataires']


## 4. Extraction et prétraitement des données historiques

In [4]:
try:
    raw_prices = download_prices(tuple(TICKERS), START_DATE, END_DATE_EXCLUSIVE)
    prices, cleaning = clean_prices(raw_prices, strict=STRICT_DATES)
except Exception as error:
    raise RuntimeError(
        f"Téléchargement/préparation impossible : {error}. "
        "Vérifier la connexion Colab, les tickers et la période, puis relancer."
    ) from error
returns = prices.pct_change(fill_method=None).dropna(how='any')
if len(returns) <= BACKTEST_WINDOW + 20:
    raise ValueError(f"Historique insuffisant pour backtest : {len(returns)} observations; "
                     f"réduisez BACKTEST_WINDOW ({BACKTEST_WINDOW}) ou reculez START_DATE.")
print('Prix du', prices.index.min().date(), 'au', prices.index.max().date())
print('Jours de rendements :', len(returns))
print('Nettoyage:', cleaning.to_dict())
display(prices.tail())


Prix du 2019-01-02 au 2026-10-08
Jours de rendements : 1952
Nettoyage: {'initial': 1953, 'retenues': 1953, 'supprimees': 0}


Ticker,SPY,QQQ,SOXX,AAPL,NVDA,TLT,BND,GLD,EFA
Date,,,,,,,,,
2026-10-02,769.640015,749.580017,588.900024,333.690002,233.949997,77.480003,69.940002,380.140015,103.959999
2026-10-05,774.830017,756.200012,589.510010,332.890015,238.899994,77.110001,69.879997,379.549988,104.029999
2026-10-06,779.090027,759.659973,589.450012,333.630005,239.240005,77.279999,70.019997,382.269989,104.220001
2026-10-07,777.219971,757.729980,582.820007,336.670013,237.470001,77.150002,70.000000,375.880005,103.099998
2026-10-08,773.929993,747.580017,563.280029,340.420013,230.479996,77.870003,70.220001,378.619995,102.699997


## 5. Performance, volatilité, Sharpe, drawdown

In [5]:
positions, summary = portfolio_statistics(returns, W, RISK_FREE_RATE_ANNUAL)
portfolio_daily = portfolio_returns(returns, W)
wealth = INITIAL_CAPITAL_USD * (1.0 + portfolio_daily).cumprod()
dd = drawdown(portfolio_daily)
print('Métriques du portefeuille')
for key, value in summary.items():
    print(f'{key}: {value:.4f}')
display(positions.style.format({
    'Poids': '{:.1%}', 'Rendement annuel (geometrique)': '{:.2%}',
    'Volatilite annuelle': '{:.2%}',
    'Contribution vol (points/an)': '{:.2%}', 'Part du risque': '{:.2%}'
}))
fig = go.Figure()
fig.add_trace(go.Scatter(x=wealth.index, y=wealth.values, name='Valeur du portefeuille'))
fig.update_layout(title='Valeur cumulée (rééquilibrage journalier théorique)', yaxis_title='USD')
fig.show()
fig = go.Figure(go.Scatter(x=dd.index, y=dd.values*100, fill='tozeroy', name='Drawdown'))
fig.update_layout(title='Drawdown du portefeuille', yaxis_title='%')
fig.show()


Métriques du portefeuille
Rendement annualise geometrique: 0.2273
Rendement log annualise: 0.2048
Volatilite annualisee: 0.1740
Sharpe annualise: 1.0920
Drawdown maximum: -0.3088
Variance via covariance: 0.0303


,Poids,Rendement annuel (geometrique),Volatilite annuelle,Contribution vol (points/an),Part du risque
Actif,,,,,
SPY,20.0%,17.41%,19.22%,3.47%,19.97%
QQQ,10.0%,23.29%,23.87%,2.26%,13.01%
SOXX,10.0%,37.02%,37.57%,3.35%,19.27%
AAPL,10.0%,32.97%,30.64%,2.34%,13.47%
NVDA,10.0%,72.55%,50.37%,4.30%,24.68%
TLT,15.0%,-2.82%,15.93%,0.25%,1.43%
BND,10.0%,1.45%,6.11%,0.21%,1.18%
GLD,10.0%,15.83%,17.80%,0.50%,2.88%
EFA,5.0%,10.70%,18.32%,0.72%,4.12%


## 6. Analyse des corrélations et décomposition des contributions au risque

Cette analyse permet de visualiser le degré de diversification effective du portefeuille et de mesurer la contribution marginale de chaque actif à la volatilité globale de l'allocation.

In [6]:
correlation = returns.corr()
covariance_annual = returns.cov() * TRADING_DAYS
display(correlation.round(2))
fig = px.imshow(correlation, text_auto='.2f', zmin=-1, zmax=1,
                color_continuous_scale='RdBu_r', title='Matrice de corrélation')
fig.show()
fig = px.bar(positions.reset_index(), x='Actif', y='Part du risque',
             title='Contribution proportionnelle à la volatilité totale')
fig.show()
print('Somme des parts de risque :', round(float(positions['Part du risque'].sum()), 6))


Ticker,SPY,QQQ,SOXX,AAPL,NVDA,TLT,BND,GLD,EFA
Ticker,,,,,,,,,
SPY,1.00,0.93,0.80,0.75,0.70,-0.12,0.18,0.13,0.85
QQQ,0.93,1.00,0.88,0.77,0.79,-0.07,0.18,0.15,0.76
SOXX,0.80,0.88,1.00,0.59,0.78,-0.07,0.13,0.16,0.70
AAPL,0.75,0.77,0.59,1.00,0.55,-0.08,0.14,0.07,0.59
NVDA,0.70,0.79,0.78,0.55,1.00,-0.05,0.11,0.10,0.55
TLT,-0.12,-0.07,-0.07,-0.08,-0.05,1.00,0.80,0.23,-0.07
BND,0.18,0.18,0.13,0.14,0.11,0.80,1.00,0.30,0.24
GLD,0.13,0.15,0.16,0.07,0.10,0.23,0.30,1.00,0.25
EFA,0.85,0.76,0.70,0.59,0.55,-0.07,0.24,0.25,1.00


Somme des parts de risque : 1.0


## 7. Estimation comparative des mesures de risques (VaR & Expected Shortfall)

Calcul comparatif de la VaR et de l'Expected Shortfall selon les approches historiques, analytiques (paramétriques) et stochastiques (simulations gaussiennes de Monte Carlo).

In [7]:
rng = np.random.default_rng(RANDOM_SEED)
historical_scenarios = historical_horizon_returns(portfolio_daily, HORIZON_DAYS)
mu_p = float(portfolio_daily.mean())
sigma_p = float(portfolio_daily.std(ddof=1))
mc_scenarios = mc_multivariate(returns, W, HORIZON_DAYS, N_MONTE_CARLO, rng)
rows = []
for alpha in CONFIDENCE_LEVELS:
    cases = {
        'Historique': risk_from_returns(historical_scenarios, alpha),
        'Paramétrique': parametric_risk(mu_p, sigma_p, alpha, HORIZON_DAYS),
        'Monte Carlo': risk_from_returns(mc_scenarios, alpha),
    }
    for method, risk in cases.items():
        rows.append({'Méthode': method, 'Confiance': alpha, 'Horizon (j)': HORIZON_DAYS,
                     'VaR (%)': risk.var, 'ES (%)': risk.es,
                     'VaR (USD)': risk.var*INITIAL_CAPITAL_USD,
                     'ES (USD)': risk.es*INITIAL_CAPITAL_USD})
risk_table = pd.DataFrame(rows)
display(risk_table.style.format({'Confiance':'{:.1%}', 'VaR (%)':'{:.2%}',
                                 'ES (%)':'{:.2%}', 'VaR (USD)':'${:,.0f}',
                                 'ES (USD)':'${:,.0f}'}))
fig = px.bar(risk_table, x='Méthode', y='VaR (USD)', color='Confiance', barmode='group',
             title=f'VaR comparée ({HORIZON_DAYS} jours)',
             hover_data=['ES (USD)'])
fig.show()
fig = px.histogram(x=-mc_scenarios*INITIAL_CAPITAL_USD, nbins=90,
                   title='Distribution simulée des pertes (Monte Carlo)',
                   labels={'x':'Perte simulée (USD)', 'y':'Effectif'})
fig.show()


,Méthode,Confiance,Horizon (j),VaR (%),ES (%),VaR (USD),ES (USD)
0,Historique,95.0%,1,1.68%,2.49%,"$16,783","$24,931"
1,Paramétrique,95.0%,1,1.72%,2.17%,"$17,160","$21,741"
2,Monte Carlo,95.0%,1,1.72%,2.18%,"$17,214","$21,770"
3,Historique,97.5%,1,2.12%,3.10%,"$21,174","$31,016"
4,Paramétrique,97.5%,1,2.06%,2.48%,"$20,614","$24,756"
5,Monte Carlo,97.5%,1,2.05%,2.48%,"$20,509","$24,835"
6,Historique,99.0%,1,2.92%,4.08%,"$29,216","$40,766"
7,Paramétrique,99.0%,1,2.46%,2.83%,"$24,631","$28,346"
8,Monte Carlo,99.0%,1,2.45%,2.86%,"$24,458","$28,617"


## 8. Backtesting statistique (Test de ratio de vraisemblance de Kupiec)

Validation empirique hors-échantillon (*out-of-sample*) consistant à évaluer de manière glissante l'exactitude de la prévision de la VaR sur un historique défini, puis à confronter le taux empirique d'infraction au taux d'exception théorique.

In [8]:
bt_hist = backtest_var(returns, W, BACKTEST_CONFIDENCE, BACKTEST_WINDOW, 'Historique')
bt_norm = backtest_var(returns, W, BACKTEST_CONFIDENCE, BACKTEST_WINDOW, 'Paramétrique')
backtest_rows = []
for method, bt in [('Historique', bt_hist), ('Paramétrique', bt_norm)]:
    exceptions = int(bt['Exception'].sum())
    lr, pvalue = kupiec_test(exceptions, len(bt), BACKTEST_CONFIDENCE)
    backtest_rows.append({'Méthode':method, 'Observations':len(bt),
                          'Exceptions':exceptions, 'Taux exceptions':exceptions/len(bt),
                          'Taux attendu':1-BACKTEST_CONFIDENCE,
                          'Statistique Kupiec':lr, 'p-value':pvalue,
                          'Feu tricolore':traffic_light(bt, BACKTEST_CONFIDENCE)})
backtest_summary = pd.DataFrame(backtest_rows)
display(backtest_summary)
fig = go.Figure()
fig.add_trace(go.Scatter(x=bt_hist.index, y=bt_hist['Perte realisee']*INITIAL_CAPITAL_USD,
                         name='Perte réalisée', mode='lines'))
fig.add_trace(go.Scatter(x=bt_hist.index, y=bt_hist['VaR prevue']*INITIAL_CAPITAL_USD,
                         name='VaR historique prévue', mode='lines'))
fig.update_layout(title='Backtesting (1 jour)', yaxis_title='USD')
fig.show()


,Méthode,Observations,Exceptions,Taux exceptions,Taux attendu,Statistique Kupiec,p-value,Feu tricolore
0,Historique,1452,18,0.012397,0.01,0.782842,0.376273,Non applicable (feu tricolore standard: 250 jo...
1,Paramétrique,1452,23,0.015840,0.01,4.248615,0.039282,Non applicable (feu tricolore standard: 250 jo...


## 9. Stress Testing historique et déformations de la structure de dépendance

Analyse du comportement du portefeuille sous des conditions de marché dégradées : évaluation de scénarios de crises macroéconomiques passées et simulation analytique d'une hausse généralisée de la corrélation d'équilibre (phénomène de contagion).

In [9]:
stress_rows = []
for label, (start, end) in CRISES.items():
    result = historical_crisis(prices, W, start, end)
    if result is None:
        stress_rows.append({'Scénario':label, 'Rendement':np.nan,
                            'P&L (USD)':np.nan, 'Statut':'Indisponible : dates ou couverture insuffisantes'})
    else:
        stress_rows.append({'Scénario':label, 'Rendement':result['return'],
                            'P&L (USD)':result['return']*INITIAL_CAPITAL_USD,
                            'Statut':f"Observé {result['from'].date()} → {result['to'].date()}"})
# Shock déterministe par catégorie (hypothèses, pas prévisions)
CATEGORY_SHOCKS = {
    'Actions US': -0.35, 'ETF actions / secteurs': -0.30,
    'ETF obligataires': 0.08, 'Autres classes (ETF)': -0.10,
}
CUSTOM_SHOCKS = {t: CATEGORY_SHOCKS[CATEGORIES[t]] for t in TICKERS}
CUSTOM_SHOCKS['SOXX'] = -0.50  # choc spécifique semi-conducteurs
scenario_result = float(sum(WEIGHTS[t]*CUSTOM_SHOCKS[t] for t in TICKERS))
stress_rows.append({'Scénario':'Actions en baisse / SOXX -50% (hypothétique)',
                    'Rendement':scenario_result, 'P&L (USD)':scenario_result*INITIAL_CAPITAL_USD,
                    'Statut':'Chocs déterministes configurables'})
stress_table = pd.DataFrame(stress_rows)
display(stress_table)
# Stress sigma et corrélation (approximation analytique, pas trajectoire)
cov_daily = returns.cov().to_numpy()
base_vol = float(np.sqrt(W @ cov_daily @ W))
for rho in [0.5, 0.8, 0.95]:
    stressed_cov = stressed_correlation_cov(cov_daily, rho)
    stressed_vol = float(np.sqrt(W @ stressed_cov @ W))
    print(f'Corrélation commune {rho:.0%}: vol journalière {stressed_vol:.2%} '
          f'(base {base_vol:.2%})')
print(f'Doublement de la volatilité : {2*base_vol:.2%} / jour, sous hypothèse de covariance ×4')


,Scénario,Rendement,P&L (USD),Statut
0,Crise financiere 2008,NaN,NaN,Indisponible : dates ou couverture insuffisantes
1,Covid 2020,-0.190296,-190296.207045,Observé 2020-02-19 → 2020-03-23
2,Marches 2022,-0.293370,-293369.534172,Observé 2022-01-03 → 2022-10-12
3,Actions en baisse / SOXX -50% (hypothétique),-0.215000,-215000.000000,Chocs déterministes configurables


Corrélation commune 50%: vol journalière 1.13% (base 1.10%)
Corrélation commune 80%: vol journalière 1.36% (base 1.10%)
Corrélation commune 95%: vol journalière 1.47% (base 1.10%)
Doublement de la volatilité : 2.19% / jour, sous hypothèse de covariance ×4


## 10. Exportation des jeux de données et métriques

Compilation et archivage automatique de l'ensemble des séries de rendements, résultats de stress tests, backtests et matrices de variance-covariance sous format compressé standardisé.

In [10]:
metadata = {
    'tickers': TICKERS, 'weights': WEIGHTS,
    'start_requested': START_DATE, 'end_exclusive': END_DATE_EXCLUSIVE,
    'sample_start': str(prices.index.min().date()), 'sample_end': str(prices.index.max().date()),
    'initial_capital_usd': INITIAL_CAPITAL_USD, 'confidence_levels': CONFIDENCE_LEVELS,
    'horizon_days': HORIZON_DAYS, 'monte_carlo_scenarios': N_MONTE_CARLO,
    'random_seed': RANDOM_SEED, 'annual_risk_free_rate_assumption': RISK_FREE_RATE_ANNUAL,
    'rebalancing': 'constant daily target weights', 'currency':'USD'
}
output_path = 'portfolio_risk_results_SOXX.zip'
with zipfile.ZipFile(output_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    files = {
        'prix_ajustes_USD.csv': prices, 'rendements_actifs.csv':returns,
        'rendements_portefeuille.csv':portfolio_daily.to_frame(),
        'positions_risque.csv':positions, 'correlations.csv':correlation,
        'covariance_annuelle.csv':covariance_annual, 'var_expected_shortfall.csv':risk_table,
        'backtesting_historique.csv':bt_hist, 'backtesting_parametrique.csv':bt_norm,
        'backtesting_resume.csv':backtest_summary, 'stress_tests.csv':stress_table,
    }
    for name, dataframe in files.items():
        z.writestr(name, dataframe.to_csv(index=True).encode('utf-8-sig'))
    z.writestr('configuration.json', json.dumps(metadata, indent=2, ensure_ascii=False))
print(f'Export créé : {output_path}')
try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print('Hors Google Colab : récupérez le fichier dans le répertoire courant.')


Export créé : portfolio_risk_results_SOXX.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Limites méthodologiques et discussion

* **Hypothèse de normalité :** L'approche paramétrique et les simulations de Monte Carlo standard reposent sur une hypothèse de distribution gaussienne multivariée, qui tend à sous-estimer la fréquence et l'amplitude des événements extrêmes (phénomène de queues épaisses ou *fat tails*).
* **Instabilité des corrélations :** La matrice des corrélations est supposée stationnaire sur la période historique, alors que les périodes de stress de marché se caractérisent souvent par une hausse brutale de la corrélation d'actifs, réduisant ainsi les effets de diversification.
* **Échantillonnage historique et disponibilité :** L'évaluation de scénarios de stress de long terme (tel que la crise de 2008) dépend de la profondeur historique des données et de l'existence effective des véhicules d'investissement à cette date.
* **Hypothèse opérationnelle :** Le modèle suppose une liquidité parfaite des actifs financiers et fait abstraction des coûts d'exécution, des taxes de transaction, ainsi que des spreads acheteur-vendeur.